<a href="https://colab.research.google.com/github/beyzadurdu6619/TrustLLM-Uncertainty-Quantification/blob/main/notebooks/06_week/temperature_scaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim


class TemperatureScaler(nn.Module):
    """TR: Logitleri T parametresine bölerek kalibre eden PyTorch sınıfı.

    EN: PyTorch module that calibrates logits using the Temperature scaling
    parameter (T).
    """

    def __init__(self):
        super(TemperatureScaler, self).__init__()
        # TR: T parametresini 1.5 olarak başlatıyoruz (Öğrenilebilir parametre).
        # EN: Initialize temperature parameter T to 1.5 (Learnable parameter).
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        # TR: Logit değerlerini T sıcaklık parametresine bölüyoruz.
        # EN: Scale the raw logits by dividing them by temperature T.
        return logits / self.temperature

    def fit(self, valid_logits, valid_labels):
        """TR: Validation seti üzerinde en uygun T değerini optimizasyonla bulur.

        EN: Finds optimal T parameter on validation set using L-BFGS optimizer.
        """
        criterion = nn.CrossEntropyLoss()

        # TR: L-BFGS algoritması sıcaklık ölçekleme için standarttır.
        # EN: Use L-BFGS optimizer which is standard for Temperature Scaling.
        optimizer = optim.LBFGS([self.temperature], lr=0.01, max_iter=50)

        def eval_loss():
            optimizer.zero_grad()
            loss = criterion(self.forward(valid_logits), valid_labels)
            loss.backward()
            return loss

        optimizer.step(eval_loss)
        print(f"Optimal Temperature (T): {self.temperature.item():.4f}")

In [2]:
import torch.nn.functional as F


def enable_dropout(model):
    """TR: Çıkarım (inference) sırasında Dropout katmanlarını açık tutar.

    EN: Forces all Dropout layers in the model to remain active during
    evaluation.
    """
    for m in model.modules():
        if m.__class__.__name__.startswith("Dropout"):
            m.train()


def estimate_mc_uncertainty(model, x_input, num_samples=20):
    """TR: MC Dropout kullanarak ortalama olasılık ve Epistemic belirsizlik (varyans) hesaplar.

    EN: Computes mean probability and epistemic uncertainty (variance) using Monte Carlo Dropout.
    """
    model.eval()
    # TR: Evaluation modunda olsak bile Dropout'ları aktif tutuyoruz.
    # EN: Force dropout layers to stay active during evaluation.
    enable_dropout(model)

    mc_predictions = []

    with torch.no_grad():
        for _ in range(num_samples):
            # TR: Açık dropout ile N kez ileri besleme yapıyoruz.
            # EN: Pass input N times with randomized dropout masks.
            logits = model(x_input)
            probs = F.softmax(logits, dim=-1)
            mc_predictions.append(probs)

    # TR: Örnekleri tek bir tensor haline getiriyoruz -> (num_samples, batch_size, num_classes)
    # EN: Stack predictions -> (num_samples, batch_size, num_classes)
    mc_predictions = torch.stack(mc_predictions)

    # TR: Ortalamasını alarak nihai olasılığı elde ediyoruz.
    # EN: Compute mean probability prediction across all MC forward passes.
    mean_probs = torch.mean(mc_predictions, dim=0)

    # TR: Tahminler arasındaki VARYANS bize Epistemic (Model) Belirsizliği verir.
    # EN: Variance across MC passes represents Epistemic (Model) Uncertainty.
    epistemic_uncertainty = torch.var(mc_predictions, dim=0)

    return mean_probs, epistemic_uncertainty